# Part 1 — Ingest, validate, clean, enrich

**Task 1.1 — CSV module exercise.** This uses csv.DictReader and a simple row converter. Run on original data file.

In [ ]:
import csv
from pathlib import Path
DATA=Path('data/appointments.csv')
def parse_row(row):
    # Converts selected numeric fields, retaining None for missing values.
    for key in ('age','fee_inr','distance_km','prior_no_shows'):
        if row.get(key) in ('',None): row[key]=None
        else: row[key]=float(row[key]) if key in ('age','distance_km') else int(float(row[key]))
    return row
with DATA.open(encoding='utf-8', newline='') as f:
    reader=csv.DictReader(f)
    headers=reader.fieldnames
    rows=[parse_row(r) for r in reader]
print('Headers:',headers)
print('Rows:',len(rows))
print(rows[0])

**Tasks 1.2–1.3 — Pandas cleaning.** Duplicates are removed; categories trimmed/canonicalized; dates parsed; invalid date/status records rejected; missing age/distance imputed with medians; numeric outliers clipped using IQR fences. Check rejected rows before using downstream.

In [ ]:
import pandas as pd, numpy as np
appointments=pd.read_csv('data/appointments.csv')
print('Raw shape:',appointments.shape)
print('Missing values:\n',appointments.isna().sum())
print('Duplicates:',appointments.duplicated().sum())
raw=appointments.drop_duplicates().copy()
raw['booking_channel']=raw.booking_channel.astype(str).str.strip().str.title()
raw['status']=raw.status.astype(str).str.strip().str.title().replace({'Nan':np.nan})
raw['booked_on']=pd.to_datetime(raw.booked_on,errors='coerce')
raw['appt_date']=pd.to_datetime(raw.appt_date,errors='coerce')
reject_mask=raw.appt_date.isna() | raw.booked_on.isna() | raw.status.isna()
rejected_rows=raw[reject_mask].copy()
clean=raw[~reject_mask].copy()
clean['age']=pd.to_numeric(clean.age,errors='coerce')
clean['distance_km']=pd.to_numeric(clean.distance_km,errors='coerce')
clean['age']=clean.age.fillna(round(clean.age.median())).clip(0,100)
clean['distance_km']=clean.distance_km.fillna(round(clean.distance_km.median(),1))
for col in ['age','distance_km','fee_inr']:
    q1,q3=clean[col].quantile([.25,.75]); iqr=q3-q1
    clean[col]=clean[col].clip(q1-1.5*iqr,q3+1.5*iqr)
clean['lead_days']=(clean.appt_date-clean.booked_on).dt.days.clip(lower=0)
clean['no_show']=(clean.status=='No-Show').astype(int)
print('Clean shape:',clean.shape,'Rejected:',len(rejected_rows))
print('Status counts:',clean.status.value_counts())
rejected_rows.to_json('rejected_rows.json',orient='records',date_format='iso')
clean['booked_on']=clean.booked_on.dt.strftime('%Y-%m-%d'); clean['appt_date']=clean.appt_date.dt.strftime('%Y-%m-%d')
clean=clean[list(appointments.columns)+['lead_days','no_show']]
clean.to_csv('appointments_clean.csv',index=False)
clean.head()

**Task 1.4 — NumPy features and summary.** Use days between booking and appointment and no-show flag; summarize distance, fee, and lead days. No Python loops for feature creation.

In [ ]:
x=clean[['distance_km','fee_inr','lead_days']].to_numpy()
print('Means:',np.mean(x,axis=0))
print('Medians:',np.median(x,axis=0))
print('Count lead_days >14:',(clean.lead_days>14).sum())
print('Count distance_km >=2:',(clean.distance_km>=2).sum())

**Task 1.5 — Clinic weather enrichment.** Free Open-Meteo forecast API example below; requires network access. Save response temperatures to clinics_enriched.csv. If it fails twice, document the clinic and leave temp_c blank (SQL NULL).

In [ ]:
import requests, time
clinics=pd.read_csv('data/clinics.csv')
clinics['temp_c']=np.nan
for idx,row in clinics.iterrows():
    url='https://api.open-meteo.com/v1/forecast'
    params={'latitude':row.latitude,'longitude':row.longitude,'hourly':'temperature_2m','forecast_days':1}
    try:
        resp=requests.get(url,params=params,timeout=15)
        if resp.status_code==200:
            payload=resp.json(); clinics.loc[idx,'temp_c']=payload['hourly']['temperature_2m'][0]
        else: print('HTTP failure:',row.clinic_id,resp.status_code)
    except Exception as e: print('API error:',row.clinic_id,e)
    time.sleep(1)
clinics.to_csv('clinics_enriched.csv',index=False)
clinics

**Task 1.6 — SQL load.** Export appointments_clean.csv with 16 columns. Run `load_data.sql` in MySQL after setting up the database; import the CSV using MySQL Workbench Table Data Import Wizard.

In [ ]:
print('CSV written:',len(clean),'rows,',len(clean.columns),'columns')
print('Appointments file: appointments_clean.csv')